# Объявление библиотек

In [2]:
import polars as pl
import pandas as pd

# Объявление функций

In [3]:
# функция используется для заданий: 1. Загрузка и первичный анализ, 6. Распределение по протоколам

# аргументы функции:
# df_lazy - lazy df
# c - столбец, для которого определяются уникальные значения и их количество
# postfix - добавление постфикса к создаваемым в функции столбцам
def print_unique_counts(df_lazy : pl.LazyFrame, c : str, c_postfix : str = "") -> pl.DataFrame:
    result = df_lazy.select(
        pl.col(c).unique(maintain_order=True).alias(f"{c} unique{c_postfix}"),
        pl.col(c).unique_counts().alias(f"{c} counts{c_postfix}")
    ).collect()

    return result

In [4]:
# функция используется для задания 8. (Бонус) Эвристика детектирования
# аргументы функции:
# df_is_suspicious - lazy df с добавленным столбцом/признаком is_suspicious
# filter_col - столбец, по которому станаывливается фильтр (по факту is_suspicious или Label
# filter_col_value - значение, по которому устанавливается фильтр(равенство ==) (по факту 0 или 1)
# check_col - столбец, который используется для расчёта (расчёт доли в столбце фильтре)
# check_tf - определение типа возвращаемого значения (True расчёт доли, False расчёт 100 минус доля)


#  пользуясь тем, что проверяем атаки (1) в числителе используем сумм, 
# count в знаменателе определяется фильтром filter_col_value (1 или 0)

# возвращаемое значение определяется check_tf (True расчёт доли, False расчёт 100 минус доля) 


def check_is_suspicious(
    df_is_suspicious : pl.LazyFrame, filter_col : str, 
    filter_col_value : int, 
    check_col : str, 
    check_tf : bool
    ) -> float :

    procent_ft = (
        df_is_suspicious
        .filter(pl.col(filter_col) == filter_col_value) # простановка фильтра
        .select(
            ( pl.col(check_col).sum() / pl.col(filter_col).count() ).round(4) # расчёт доли true positives и пр.
        )
    ).collect()[0, 0] * 100 # использую 100, для процентов

    if check_tf:         
        return procent_ft
    else:
        return round(100 - procent_ft, 4) # расчёт доли false positives и пр. (использую 100, для процентов) 

# 1. Загрузка и первичный анализ

## Загрузите файл NF-CSE-CIC-IDS2018-V2.parquet в Polars DataFrame

In [35]:
# df загружаем в lazy режиме
df = pl.scan_parquet("NF-CSE-CIC-IDS2018-V2.parquet")

## Первичный анализ

In [37]:
# смотрим первые 5 и последние 5 строк (видим названия признаков/столбцов и их тип данных)
df_concat = pl.concat(
        ( df.head(5).collect(), df.tail(5).collect() )
        )
display(df_concat)

L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
i32,i32,i8,f32,i32,i32,i32,i32,i16,i16,i16,i32,i32,i32,i16,i16,i32,i16,i16,i32,f64,f64,i32,i16,i32,i16,i64,i64,i32,i16,i16,i16,i32,i32,i32,i32,i16,i32,i16,i32,i8,i8,str
40894,22,6,92.0,3164,23,3765,21,27,27,27,0,0,0,63,63,1028,52,52,1028,3164.0,3765.0,0,0,0,0,25312000,30120000,33,7,1,2,1,26883,26847,0,0,0,0,0,0,1,"""SSH-Bruteforce"""
29622,3389,6,0.0,1919,14,2031,11,223,219,30,0,0,0,101,101,1195,40,40,1195,1919.0,2031.0,0,0,0,0,15352000,16248000,17,6,0,1,1,8192,64000,0,0,0,0,0,0,0,"""Benign"""
65456,53,17,0.0,116,2,148,2,0,0,0,0,0,0,128,128,74,58,58,74,116.0,148.0,0,0,0,0,928000,1184000,4,0,0,0,0,0,0,0,0,2511,1,5,0,0,"""Benign"""
57918,53,17,0.0,70,1,130,1,0,0,0,0,0,0,0,0,130,70,70,130,70.0,130.0,0,0,0,0,560000,1040000,1,1,0,0,0,0,0,0,0,3371,1,60,0,0,"""Benign"""
63269,80,6,7.0,232,5,1136,4,223,222,27,4294827,140,0,127,127,1004,40,40,1004,232.0,1136.0,0,0,0,0,8000,9088000,8,0,0,1,0,8192,26883,0,0,0,0,0,0,1,"""DDoS attacks-LOIC-HTTP"""
59578,53,17,0.0,77,1,138,1,0,0,0,0,0,0,0,0,138,77,77,138,77.0,138.0,0,0,0,0,616000,1104000,1,1,0,0,0,0,0,0,0,14428,1,60,0,0,"""Benign"""
52954,443,6,91.220001,1729,20,6719,16,27,27,27,0,0,0,128,128,1500,40,40,1500,1729.0,6719.0,0,0,0,0,13832000,53752000,26,3,3,1,3,8192,29200,0,0,0,0,0,0,0,"""Benign"""
59664,80,6,7.0,561,5,1147,5,219,219,27,4294937,29,29,127,127,975,40,40,975,561.0,1147.0,0,0,0,0,144000,304000,8,0,1,1,0,65535,26883,0,0,0,0,0,0,1,"""DDOS attack-HOIC"""
1780,445,6,41.0,120,3,124,3,23,19,22,0,0,0,106,106,44,40,40,44,120.0,124.0,0,0,0,0,960000,992000,6,0,0,0,0,65392,65392,0,0,0,0,0,0,0,"""Benign"""


In [38]:
# смотрим распределение для выборочных 100 000 строк
display(df.collect().sample(100_000).describe())

statistic,L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""count""",100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,"""100000"""
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""0"""
"""mean""",52912.49608,1086.56826,11.03142,11.249846,1990.70191,26.0179,4880.67668,8.54968,72.59109,71.91145,20.40388,599893.7918,27.04996,12.43982,69.75919,69.7919,572.16732,54.41114,51.72401,572.16732,8.8889e194,2.0878e12,40.27633,0.53699,59.47418,0.12609,4.8086e6,1.4256e7,29.0134,1.42573,0.77191,0.6268,2.82984,11514.18524,20839.49081,3262.68234,12.74485,14889.76712,4.39169,21.21827,0.0,0.11678,null
"""std""",11586.056626,3240.167285,5.49329,26.958912,76059.300702,1267.006824,247396.162464,168.167417,97.944225,97.727763,28.403564,1.4888e6,1161.519439,78.862877,56.765592,56.78587,543.036721,33.599226,18.221043,543.036721,inf,6.5841e14,299.77551,2.993894,3910.522615,2.933823,9.3783e6,9.3700e7,1267.308064,8.619667,4.657255,4.583057,164.634528,19428.336297,25393.240821,12084.793377,47.206225,20738.50145,16.677709,26.930358,0.0,0.32116,null
"""min""",0.0,0.0,1.0,0.0,28.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,28.0,28.0,0.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Benign"""
"""25%""",50901.0,53.0,6.0,0.0,69.0,1.0,119.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,106.0,40.0,40.0,106.0,70.0,125.0,0.0,0.0,0.0,0.0,528000.0,776000.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null
"""50%""",54575.0,80.0,6.0,0.0,140.0,3.0,280.0,3.0,23.0,19.0,22.0,0.0,0.0,0.0,100.0,100.0,183.0,44.0,40.0,183.0,154.0,310.0,0.0,0.0,0.0,0.0,960000.0,1.352e6,5.0,1.0,0.0,0.0,0.0,8192.0,8192.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null
"""75%""",59787.0,445.0,17.0,5.7,1460.0,9.0,1873.0,8.0,219.0,219.0,27.0,0.0,0.0,0.0,127.0,127.0,1189.0,67.0,67.0,1189.0,1464.0,1873.0,0.0,0.0,0.0,0.0,1.0592e7,1.3768e7,12.0,2.0,1.0,1.0,1.0,8192.0,30016.0,0.0,0.0,29542.0,1.0,60.0,0.0,0.0,null
"""max""",65535.0,65129.0,58.0,245.0,1.135194e7,189199.0,5.8243831e7,39026.0,223.0,223.0,223.0,4.294967e6,105622.0,6545.0,255.0,255.0,2951.0,1478.0,328.0,2951.0,8.8889e199,2.0821e17,22784.0,149.0,1.045861e6,705.0,7.64448e8,4.2936e9,189199.0,1508.0,443.0,1189.0,38769.0,65535.0,65535.0,65280.0,255.0,65535.0,255.0,60.0,0.0,1.0,"""SSH-Bruteforce"""


In [ ]:
# Дополнительно. Информация уже по всему df
display(df.collect_schema())

Schema([('L4_SRC_PORT', Int32),
        ('L4_DST_PORT', Int32),
        ('PROTOCOL', Int8),
        ('L7_PROTO', Float32),
        ('IN_BYTES', Int32),
        ('IN_PKTS', Int32),
        ('OUT_BYTES', Int32),
        ('OUT_PKTS', Int32),
        ('TCP_FLAGS', Int16),
        ('CLIENT_TCP_FLAGS', Int16),
        ('SERVER_TCP_FLAGS', Int16),
        ('FLOW_DURATION_MILLISECONDS', Int32),
        ('DURATION_IN', Int32),
        ('DURATION_OUT', Int32),
        ('MIN_TTL', Int16),
        ('MAX_TTL', Int16),
        ('LONGEST_FLOW_PKT', Int32),
        ('SHORTEST_FLOW_PKT', Int16),
        ('MIN_IP_PKT_LEN', Int16),
        ('MAX_IP_PKT_LEN', Int32),
        ('SRC_TO_DST_SECOND_BYTES', Float64),
        ('DST_TO_SRC_SECOND_BYTES', Float64),
        ('RETRANSMITTED_IN_BYTES', Int32),
        ('RETRANSMITTED_IN_PKTS', Int16),
        ('RETRANSMITTED_OUT_BYTES', Int32),
        ('RETRANSMITTED_OUT_PKTS', Int16),
        ('SRC_TO_DST_AVG_THROUGHPUT', Int64),
        ('DST_TO_SRC_AVG_THROUGHPU

## Количество строк и колонок

In [41]:
# сортируем общее количество строк (для удобства с разделителями) и столбцов 
print( 
      "Количество строк - {0:_.0f} \nКоличество столбцов - {1}"\
          .format(df.collect().shape[0], df.collect().shape[1])\
              .replace("_", " ") 
      )

Количество строк - 17 129 715 
Количество столбцов - 43


## Типы колонок (последние пять)

In [42]:
display(
    pl.DataFrame(
            {
                "Столбец" : df.collect_schema().names(),
                "Тип столбца" : df.collect_schema().dtypes()
                }
        )[-5:] # нужно вывести последние пять столбцов

    )

Столбец,Тип столбца
str,object
"""DNS_QUERY_TYPE""",Int16
"""DNS_TTL_ANSWER""",Int32
"""FTP_COMMAND_RET_CODE""",Int8
"""Label""",Int8
"""Attack""",String


## Уникальные значения в колонке Label и уникальные значения в колонке Attack

In [ ]:
cols = ["Label", "Attack"]

cols_name = [] # для имён столбцов
cols_unique = [] # для уникальных элементах в столбцах 
cols_unique_counts = [] # для подсчёта уникальных элементов

# заполнение списков
for c in cols:
     # для создания столбца в df, соответствующего количеству его уникальных значений len( df.collect()[c].unique() )
    cols_name.extend([c] * len( df.collect()[c].unique() )  )
    cols_unique.extend( df.collect()[c].unique(maintain_order=True) )
    cols_unique_counts.extend( df.collect()[c].unique_counts() )

# для удобства вывод как df (типы в столбце cols_unique разные, поэтому pandas)
df_pandas = pd.DataFrame(
        {
            'Название столбца' : cols_name,
            'Уникальные элементы столбца' : cols_unique,
            'Количество уникальных элементов' : cols_unique_counts
        }
    )

display(df_pandas)

,Название столбца,Уникальные элементы столбца,Количество уникальных элементов
0,Label,1,2028030
1,Label,0,15101685
2,Attack,SSH-Bruteforce,94979
3,Attack,Benign,15101685
4,Attack,DDoS attacks-LOIC-HTTP,207078
5,Attack,DDOS attack-HOIC,1066881
6,Attack,DoS attacks-Slowloris,9512
7,Attack,DoS attacks-Hulk,432648
8,Attack,FTP-BruteForce,25933
9,Attack,Infilteration,115513


# 2. Распределение по меткам

In [13]:
# создание переменных для вывода доли

# сколько записей помечено как Label == 0 ("Benign")
counts_0 = (
        df
        .filter(pl.col("Label") == 0)
        .select(pl.col("Label").unique_counts())
    ).collect()[0, 0]

# сколько записей помечено как Label == 1 ("Attack")
counts_1 = (
        df
        .filter(pl.col("Label") == 1)
        .select(pl.col("Label").unique_counts())
    ).collect()[0, 0]

# для удобства вывод в процентах
prop_benign =  "{}%".format(round( counts_0 / (counts_0 + counts_1) * 100, 2 ))
prop_attack =  "{}%".format( round( counts_1 / (counts_0 + counts_1) * 100, 2 ) )
# для удобства  вывод как df
display(
pl.DataFrame({
    "Название" : ["Пропорция Benign", "Пропорция Attack"],
    "Пропорция" : [prop_benign, prop_attack]
    })
)

Название,Пропорция
str,str
"""Пропорция Benign""","""88.16%"""
"""Пропорция Attack""","""11.84%"""


# 3. Создание бинарного признака

Добавьте колонку is_attack, которая: 

1, если Label != 0 (атака) 

0, если Label == 0 (норма)

In [14]:
df = df.with_columns(
    pl.when(pl.col("Label") != 0)
        .then(1)
        .when(pl.col("Label") == 0)
        .then(0)
        .alias("is_attack")        
    )
df = df.cast({"is_attack" : pl.Int8})
print(df.collect())

shape: (17_129_715, 44)
┌────────────┬────────────┬──────────┬───────────┬───┬────────────┬───────┬────────────┬───────────┐
│ L4_SRC_POR ┆ L4_DST_POR ┆ PROTOCOL ┆ L7_PROTO  ┆ … ┆ FTP_COMMAN ┆ Label ┆ Attack     ┆ is_attack │
│ T          ┆ T          ┆ ---      ┆ ---       ┆   ┆ D_RET_CODE ┆ ---   ┆ ---        ┆ ---       │
│ ---        ┆ ---        ┆ i8       ┆ f32       ┆   ┆ ---        ┆ i8    ┆ str        ┆ i8        │
│ i32        ┆ i32        ┆          ┆           ┆   ┆ i8         ┆       ┆            ┆           │
╞════════════╪════════════╪══════════╪═══════════╪═══╪════════════╪═══════╪════════════╪═══════════╡
│ 40894      ┆ 22         ┆ 6        ┆ 92.0      ┆ … ┆ 0          ┆ 1     ┆ SSH-Brutef ┆ 1         │
│            ┆            ┆          ┆           ┆   ┆            ┆       ┆ orce       ┆           │
│ 29622      ┆ 3389       ┆ 6        ┆ 0.0       ┆ … ┆ 0          ┆ 0     ┆ Benign     ┆ 0         │
│ 65456      ┆ 53         ┆ 17       ┆ 0.0       ┆ … ┆ 0          ┆

# 4. Агрегация по типам атак

In [15]:
# по заданию
# Отфильтруйте только атаки (Label != 0).
# Сгруппируйте данные по колонке Attack.
# Посчитайте:
# среднюю длительность потока (FLOW_DURATION_MILLISECONDS);
# среднее количество входящих байт (IN_BYTES);
# общее количество записей для каждого типа атаки.
# Отсортируйте по убыванию avg_in_bytes.

df_agg_attack = (
    df
    .filter(pl.col("Label") != 0)
    .group_by("Attack")
    .agg([
        pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_flow_duration"),
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.col("Attack").count().alias("count_values")
        ])
    .sort(by=pl.col("avg_in_bytes"), descending=True)
    .collect()
    )

display(df_agg_attack)

Attack,avg_flow_duration,avg_in_bytes,count_values
str,f64,f64,u32
"""DDOS attack-LOIC-UDP""",4.1959e6,5.8540e6,2112
"""DDoS attacks-LOIC-HTTP""",3.7241e6,25991.904925,207078
"""Brute Force -XSS""",3.7568e6,16871.281553,927
"""Brute Force -Web""",3.7553e6,9797.653756,2143
"""SSH-Bruteforce""",733291.768981,5828.140747,94979
…,…,…,…
"""Infilteration""",234910.506514,791.858613,115513
"""Bot""",1.2340e6,677.354225,28033
"""SQL Injection""",3.7282e6,599.328704,432


In [16]:
# Сохраните итоговую агрегацию (по типам атак) в файл attack_summary_by_type.parquet.
df_agg_attack.write_parquet("attack_summary_by_type.parquet")

# 5. Топ-3 атак по трафику

In [17]:
print("Топ-3 атак по трафику")
print(df_agg_attack[:3])

Топ-3 атак по трафику
shape: (3, 4)
┌────────────────────────┬───────────────────┬──────────────┬──────────────┐
│ Attack                 ┆ avg_flow_duration ┆ avg_in_bytes ┆ count_values │
│ ---                    ┆ ---               ┆ ---          ┆ ---          │
│ str                    ┆ f64               ┆ f64          ┆ u32          │
╞════════════════════════╪═══════════════════╪══════════════╪══════════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6          ┆ 5.8540e6     ┆ 2112         │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6          ┆ 25991.904925 ┆ 207078       │
│ Brute Force -XSS       ┆ 3.7568e6          ┆ 16871.281553 ┆ 927          │
└────────────────────────┴───────────────────┴──────────────┴──────────────┘


# 6. Распределение по протоколам

## Общее распределение по PROTOCOL

In [18]:
df.select(pl.col("PROTOCOL")).collect().describe()

statistic,PROTOCOL
str,f64
"""count""",1.7129715e7
"""null_count""",0.0
"""mean""",10.994813
"""std""",5.488639
"""min""",1.0
"""25%""",6.0
"""50%""",6.0
"""75%""",17.0
"""max""",58.0


In [19]:
print_unique_counts(df, "PROTOCOL")

PROTOCOL unique,PROTOCOL counts
i8,u32
6,9346287
17,7776756
1,4857
2,976
58,836
47,3


## Распределение по PROTOCOL только для Benign

In [20]:
(
    df
    .filter(pl.col("Label") == 0)
    .select(pl.col("PROTOCOL"))
    .collect()
    .describe()
)

statistic,PROTOCOL
str,f64
"""count""",1.5101685e7
"""null_count""",0.0
"""mean""",11.601441
"""std""",5.512306
"""min""",1.0
"""25%""",6.0
"""50%""",17.0
"""75%""",17.0
"""max""",58.0


In [21]:
df_label_0 = print_unique_counts(df.filter(pl.col("Label") == 0), "PROTOCOL", "_L_0")
print("BENIGN")
display(df_label_0)

BENIGN


PROTOCOL unique_L_0,PROTOCOL counts_L_0
i8,u32
6,7406897
17,7688529
1,4537
2,883
58,836
47,3


## Распределение по PROTOCOL только для Attack (с разбивкой по Attack)

In [22]:
(
    df
    .filter(pl.col("Label") != 0)
    .select(pl.col("PROTOCOL"))
    .collect()
    .describe()
)

statistic,PROTOCOL
str,f64
"""count""",2.02803e6
"""null_count""",0.0
"""mean""",6.477569
"""std""",2.245121
"""min""",1.0
"""25%""",6.0
"""50%""",6.0
"""75%""",6.0
"""max""",17.0


In [23]:
# Распределение по PROTOCOL только для Attack (с разбивкой по Attack)
df_attack_protocol = (
    df
    .filter(pl.col("Label") != 0)
    .group_by(["Attack", "PROTOCOL"])
    .agg([
        pl.col("PROTOCOL").count().alias("PROTOCOL counts")
        ])
    .sort(by=pl.col(["PROTOCOL", "PROTOCOL counts"]))
    .collect()
    )

print(df_attack_protocol)

shape: (20, 3)
┌────────────────────────┬──────────┬─────────────────┐
│ Attack                 ┆ PROTOCOL ┆ PROTOCOL counts │
│ ---                    ┆ ---      ┆ ---             │
│ str                    ┆ i8       ┆ u32             │
╞════════════════════════╪══════════╪═════════════════╡
│ Infilteration          ┆ 1        ┆ 320             │
│ Infilteration          ┆ 2        ┆ 93              │
│ SQL Injection          ┆ 6        ┆ 432             │
│ Brute Force -XSS       ┆ 6        ┆ 880             │
│ Brute Force -Web       ┆ 6        ┆ 2079            │
│ …                      ┆ …        ┆ …               │
│ Brute Force -XSS       ┆ 17       ┆ 47              │
│ Brute Force -Web       ┆ 17       ┆ 64              │
│ DDOS attack-LOIC-UDP   ┆ 17       ┆ 2112            │
│ DDoS attacks-LOIC-HTTP ┆ 17       ┆ 17328           │
│ Infilteration          ┆ 17       ┆ 68676           │
└────────────────────────┴──────────┴─────────────────┘


## Cравнение PROTOCOL между Benign и Attack

In [24]:
df_label_1 = print_unique_counts(df.filter(pl.col("Label") != 0), "PROTOCOL", "_L_1")
print("ATTACK")
display(df_label_1)

ATTACK


PROTOCOL unique_L_1,PROTOCOL counts_L_1
i8,u32
6,1939390
17,88227
2,93
1,320


In [25]:
# соединение BENIGN и ATTACK
# использую определённые ранее префиксы 
# L_0 (Label == 0 Benign)
#  L_0 (Label == 1 Attack)
# здесь правильней сделать полное соединение и заполнить null НУЛЯМИ

full_join = df_label_0.join(
    df_label_1, 
    left_on="PROTOCOL unique_L_0",
    right_on="PROTOCOL unique_L_1",
    how="full"
    ).select(
        pl.col("PROTOCOL unique_L_0"),
        pl.col("PROTOCOL counts_L_0").fill_null(0),
        pl.col("PROTOCOL counts_L_1").fill_null(0),     
        )

full_join = full_join.with_columns(
    (  100 * pl.col("PROTOCOL counts_L_1") / ( pl.col("PROTOCOL counts_L_0") + pl.col("PROTOCOL counts_L_1") )  ).round(2)
    .alias("Процент доли Атак в Протоколе")
    )

display(full_join)

PROTOCOL unique_L_0,PROTOCOL counts_L_0,PROTOCOL counts_L_1,Процент доли Атак в Протоколе
i8,u32,u32,f64
6,7406897,1939390,20.75
17,7688529,88227,1.13
1,4537,320,6.59
2,883,93,9.53
58,836,0,0.0
47,3,0,0.0


Сразу видны протоколы, где нет атак (58, 47) и много атак (6, 2, 1)

In [26]:
# доля атак по видам атаки и по протоколам
df_attack_protocol = df_attack_protocol.with_columns(
    ( 
        ( 100 * pl.col("PROTOCOL counts") / pl.col("PROTOCOL counts").sum().over(pl.col("PROTOCOL")) ).round(2)
        .alias("Процент доли вида атаки для протокола")
    )
)

display(df_attack_protocol)

Attack,PROTOCOL,PROTOCOL counts,Процент доли вида атаки для протокола
str,i8,u32,f64
"""Infilteration""",1,320,100.0
"""Infilteration""",2,93,100.0
"""SQL Injection""",6,432,0.02
"""Brute Force -XSS""",6,880,0.05
"""Brute Force -Web""",6,2079,0.11
…,…,…,…
"""Brute Force -XSS""",17,47,0.05
"""Brute Force -Web""",17,64,0.07
"""DDOS attack-LOIC-UDP""",17,2112,2.39


В 17 протоколе большая доля атак (78%) для типа Infilteration

In [27]:
display(df_attack_protocol
      .filter(
          (pl.col("PROTOCOL")==2) | (pl.col("PROTOCOL")==1) | (pl.col("PROTOCOL")==6)
          )
      )

Attack,PROTOCOL,PROTOCOL counts,Процент доли вида атаки для протокола
str,i8,u32,f64
"""Infilteration""",1,320,100.0
"""Infilteration""",2,93,100.0
"""SQL Injection""",6,432,0.02
"""Brute Force -XSS""",6,880,0.05
"""Brute Force -Web""",6,2079,0.11
…,…,…,…
"""Infilteration""",6,46424,2.39
"""SSH-Bruteforce""",6,94979,4.9
"""DDoS attacks-LOIC-HTTP""",6,189750,9.78


В 1 и 2 протоколе большая доля атак для типа Infilteration  
В 6 протоколе в основном DOS атаки

# 7. Сравнение метрик: Benign vs Attack

In [28]:
# Сгруппируйте по is_attack и посчитайте:
# средние значения IN_BYTES, OUT_BYTES, FLOW_DURATION_MILLISECONDS;
# общее количество записей

df_is_attack = (
    df
    .group_by("is_attack")
    .agg([
        pl.col("IN_BYTES").mean().alias("avg_in_bytes"),
        pl.col("OUT_BYTES").mean().alias("avg_out_bytes"),
        pl.col("FLOW_DURATION_MILLISECONDS").mean().alias("avg_flow_duration"),
        pl.col("is_attack").count().alias("count_values")    
        ])
    .collect()
    )

print(df_is_attack)

shape: (2, 5)
┌───────────┬──────────────┬───────────────┬───────────────────┬──────────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_flow_duration ┆ count_values │
│ ---       ┆ ---          ┆ ---           ┆ ---               ┆ ---          │
│ i8        ┆ f64          ┆ f64           ┆ f64               ┆ u32          │
╞═══════════╪══════════════╪═══════════════╪═══════════════════╪══════════════╡
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107     ┆ 15101685     │
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6          ┆ 2028030      │
└───────────┴──────────────┴───────────────┴───────────────────┴──────────────┘


Видно, что для атак характерно значительное превышение **avg_in_bytes** над **avg_out_bytes**

# 8. (Бонус) Эвристика детектирования

In [29]:
# Создайте колонку is_suspicious, которая равна:
# 1, если:
# bytes_ratio > 10 (входящий трафик >> исходящего);
# FLOW_DURATION_MILLISECONDS < 500 (короткий поток);
# IN_PKTS > 10 (много пакетов).
# 0 — в противном случае.


df_is_suspicious = (
    df
    .with_columns(
    pl.when(        
                ( ( pl.col("IN_BYTES") / (pl.col("OUT_BYTES") + 1) ) > 10 )
            &
                 ( pl.col("FLOW_DURATION_MILLISECONDS") < 500 )
            &
                ( pl.col("IN_PKTS") > 10 )
            )
        .then(1)
        .otherwise(0)
        .alias("is_suspicious")
        )
    )

display(
    pl.concat(
        ( df_is_suspicious.head(5).collect(), df_is_suspicious.tail(5).collect() )
        )
)

L4_SRC_PORT,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,Label,Attack,is_attack,is_suspicious
i32,i32,i8,f32,i32,i32,i32,i32,i16,i16,i16,i32,i32,i32,i16,i16,i32,i16,i16,i32,f64,f64,i32,i16,i32,i16,i64,i64,i32,i16,i16,i16,i32,i32,i32,i32,i16,i32,i16,i32,i8,i8,str,i8,i32
40894,22,6,92.0,3164,23,3765,21,27,27,27,0,0,0,63,63,1028,52,52,1028,3164.0,3765.0,0,0,0,0,25312000,30120000,33,7,1,2,1,26883,26847,0,0,0,0,0,0,1,"""SSH-Bruteforce""",1,0
29622,3389,6,0.0,1919,14,2031,11,223,219,30,0,0,0,101,101,1195,40,40,1195,1919.0,2031.0,0,0,0,0,15352000,16248000,17,6,0,1,1,8192,64000,0,0,0,0,0,0,0,"""Benign""",0,0
65456,53,17,0.0,116,2,148,2,0,0,0,0,0,0,128,128,74,58,58,74,116.0,148.0,0,0,0,0,928000,1184000,4,0,0,0,0,0,0,0,0,2511,1,5,0,0,"""Benign""",0,0
57918,53,17,0.0,70,1,130,1,0,0,0,0,0,0,0,0,130,70,70,130,70.0,130.0,0,0,0,0,560000,1040000,1,1,0,0,0,0,0,0,0,3371,1,60,0,0,"""Benign""",0,0
63269,80,6,7.0,232,5,1136,4,223,222,27,4294827,140,0,127,127,1004,40,40,1004,232.0,1136.0,0,0,0,0,8000,9088000,8,0,0,1,0,8192,26883,0,0,0,0,0,0,1,"""DDoS attacks-LOIC-HTTP""",1,0
59578,53,17,0.0,77,1,138,1,0,0,0,0,0,0,0,0,138,77,77,138,77.0,138.0,0,0,0,0,616000,1104000,1,1,0,0,0,0,0,0,0,14428,1,60,0,0,"""Benign""",0,0
52954,443,6,91.220001,1729,20,6719,16,27,27,27,0,0,0,128,128,1500,40,40,1500,1729.0,6719.0,0,0,0,0,13832000,53752000,26,3,3,1,3,8192,29200,0,0,0,0,0,0,0,"""Benign""",0,0
59664,80,6,7.0,561,5,1147,5,219,219,27,4294937,29,29,127,127,975,40,40,975,561.0,1147.0,0,0,0,0,144000,304000,8,0,1,1,0,65535,26883,0,0,0,0,0,0,1,"""DDOS attack-HOIC""",1,0
1780,445,6,41.0,120,3,124,3,23,19,22,0,0,0,106,106,44,40,40,44,120.0,124.0,0,0,0,0,960000,992000,6,0,0,0,0,65392,65392,0,0,0,0,0,0,0,"""Benign""",0,0


Подсчитайте: 

сколько записей помечено как is_suspicious == 1; 

сколько из них на самом деле являются атаками (Label != 0) 

Вычислите точность эвристики: true_attacks / total_flagged

In [30]:
print( "Процент наличия реальных аттак (Label=1) среди протоколов, которые были определены как подозрительные (is_suspicious=1) {0:.2f}%".format(
        check_is_suspicious(df_is_suspicious, "is_suspicious", 1, "Label", True)
    )
)

Процент наличия реальных аттак (Label=1) среди протоколов, которые были определены как подозрительные (is_suspicious=1) 79.81%


In [31]:
print( "Процент отсутствия реальных атак (Label=0) среди протоколов, которые были определены как подозрительные (is_suspicious=1) {0:.2f}%".format(
        check_is_suspicious(df_is_suspicious, "is_suspicious", 1, "Label", False)
    )
)

Процент отсутствия реальных атак (Label=0) среди протоколов, которые были определены как подозрительные (is_suspicious=1) 20.19%


In [32]:
print( "Процент протоколов, которые были определены как подозрительные (is_suspicious=1), среди наличия реальных атак (Label=1) {0:.2f}%".format(
        check_is_suspicious(df_is_suspicious, "Label", 1, "is_suspicious", True)
    )
)

Процент протоколов, которые были определены как подозрительные (is_suspicious=1), среди наличия реальных атак (Label=1) 0.18%


In [33]:
print( "Процент протоколов, которые не были определены как подозрительные (is_suspicious=0), среди наличия реальных атак (Label=1) {0:.2f}%".format(
        check_is_suspicious(df_is_suspicious, "Label", 1, "is_suspicious", False)
    )
)

Процент протоколов, которые не были определены как подозрительные (is_suspicious=0), среди наличия реальных атак (Label=1) 99.82%


In [34]:
print( "Процент протоколов, которые были определены как подозрительные (is_suspicious=1), среди отсутствия реальных атак (Label=0) {0:.2f}%".format(
        check_is_suspicious(df_is_suspicious, "Label", 0, "is_suspicious", True)
    )
)

Процент протоколов, которые были определены как подозрительные (is_suspicious=1), среди отсутствия реальных атак (Label=0) 0.01%


Показатель true_attacks / total_flagged около 80%. Но, судя по тому, что остальные показатели крайне низкие (Процент протоколов, которые не были определены как подозрительные (is_suspicious=0), среди наличия реальных атак (Label=1) 99.82%) даже для эвристической модели нужно более глубокое понимание характеристик атак.